In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import community as community_louvain
from sklearn.ensemble import IsolationForest
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Loading the CSV files 
train_transac = pd.read_csv("Dataset/train_transaction.csv")
test_transac = pd.read_csv("Dataset/test_transaction.csv")
train_id = pd.read_csv("Dataset/train_identity.csv")
test_id = pd.read_csv("Dataset/train_identity.csv")

# 1. Merge and Split Data Chronologically
full_train_df = pd.merge(train_transac, train_id, on='TransactionID', how='left')
full_train_df = full_train_df.sort_values('TransactionDT').reset_index(drop=True)

split_index = int(len(full_train_df) * 0.8)
df_train = full_train_df.iloc[:split_index].copy()
df_val = full_train_df.iloc[split_index:].copy()

# 2. Pre-calculate entity frequencies
print("Calculating entity frequencies for edge weighting...")
entity_frequencies = Counter()

for _, row in df_train.iterrows():
    for col in ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 
                'DeviceInfo', 'DeviceType', 'id_31', 'id_30', 'P_emaildomain']:
        if col in row and pd.notna(row[col]):
            val = int(row[col]) if col in ['card1', 'card2', 'card3', 'card5'] else row[col]
            entity_node = f"{col}_{val}"
            entity_frequencies[entity_node] += 1

# 3. Initialize the Knowledge Graph
G = nx.Graph()

# 4. Build KG with Hyper-Hub Penalized Edge Weights
hub_threshold = len(df_train) * 0.01 

for _, row in df_train.iterrows():
    tx_id = row['TransactionID']
    
    G.add_node(
        tx_id, 
        node_type='Transaction',
        amount=row.get('TransactionAmt', 0),
        is_fraud=row.get('isFraud', 0),
        timestamp=row.get('TransactionDT', 0),
        id_01=row.get('id_01', 0),
        id_02=row.get('id_02', 0)
    )
    
    for col in ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 
                'DeviceInfo', 'DeviceType', 'id_31', 'id_30', 'P_emaildomain']:
        if col in row and pd.notna(row[col]):
            val = int(row[col]) if col in ['card1', 'card2', 'card3', 'card5'] else row[col]
            entity_node = f"{col}_{val}"
            
            freq = entity_frequencies[entity_node]
            
            if freq > hub_threshold:
                weight = 0.01 / freq  # Hyper-hub penalty
            else:
                weight = 1.0 / (1.0 + np.log(freq))
            
            G.add_node(entity_node, node_type=col, frequency=freq)
            G.add_edge(tx_id, entity_node, relationship=col, weight=weight)

# 5. Extract Superior Community-Driven Topology Metrics & Print Diagnostics
print("Computing ring-optimized topological features...")
pagerank = nx.pagerank(G, alpha=0.85, weight='weight')
partition = community_louvain.best_partition(G, weight='weight')
nx.set_node_attributes(G, partition, 'community_id')

# --- COMMUNITY PARTITION DIAGNOSTICS ---
community_sizes = Counter(partition.values())
print(f"\n--- Community Partition Diagnostics ---")
print(f"Total unique communities detected: {len(community_sizes)}")
print(f"Largest community size: {max(community_sizes.values())} nodes")
print(f"Smallest community size: {min(community_sizes.values())} nodes")
print(f"Median community size: {np.median(list(community_sizes.values())):.1f} nodes")

comm_temp_df = pd.DataFrame([
    {'community_id': data.get('community_id', -1), 'is_fraud': data.get('is_fraud', 0)}
    for node, data in G.nodes(data=True) if data.get('node_type') == 'Transaction'
])

comm_summary = comm_temp_df.groupby('community_id').agg(
    total_tx=('is_fraud', 'count'),
    fraud_count=('is_fraud', 'sum'),
    fraud_rate=('is_fraud', 'mean')
).reset_index()

comm_summary = comm_summary[comm_summary['total_tx'] > 1].sort_values(by='fraud_rate', ascending=False)
print("\nTop 5 Most Suspicious Communities (by Fraud Rate):")
print(comm_summary.head(5).to_string(index=False))
print("-" * 45 + "\n")

# 6. Extract Training Feature Matrix
graph_features = []
for node, data in G.nodes(data=True):
    if data.get('node_type') == 'Transaction':
        neighbors = list(G.neighbors(node))
        
        neighbor_communities = [
            G.nodes[n].get('community_id', -1) 
            for n in neighbors if 'community_id' in G.nodes[n]
        ]
        assigned_community = max(set(neighbor_communities), key=neighbor_communities.count) if neighbor_communities else -1
        
        weighted_deg = sum(G[node][n].get('weight', 1.0) for n in neighbors)
        
        graph_features.append({
            'TransactionID': node,
            'isFraud': data.get('is_fraud', 0),
            'TransactionAmt': data.get('amount', 0),
            'id_01': data.get('id_01', 0),
            'id_02': data.get('id_02', 0),
            'graph_degree': G.degree(node),
            'weighted_degree': weighted_deg,
            'pagerank': pagerank.get(node, 0),
            'assigned_community': assigned_community
        })

df_features = pd.DataFrame(graph_features)

comm_fraud_rates = df_features.groupby('assigned_community')['isFraud'].mean().to_dict()
df_features['community_fraud_rate'] = df_features['assigned_community'].map(comm_fraud_rates).fillna(df_features['isFraud'].mean())

# 7. Train Isolation Forest with Ring-Optimized Features
feature_cols = [
    'TransactionAmt', 
    'id_01', 
    'id_02',
    'graph_degree', 
    'weighted_degree',
    'pagerank',
    'community_fraud_rate'
]

X_train = df_features[feature_cols].fillna(0)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

iso_forest = IsolationForest(
    n_estimators=150,
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train_scaled)

Calculating entity frequencies for edge weighting...
Computing ring-optimized topological features...


In [ ]:
# --------------------------- VALIDATION EVALUATION ------------------------------------
print("Extracting validation graph features...")
val_graph_features = []
for _, row in df_val.iterrows():
    tx_id = row['TransactionID']
    
    G.add_node(
        tx_id, 
        node_type='ValTransaction',
        amount=row.get('TransactionAmt', 0),
        is_fraud=row.get('isFraud', 0),
        timestamp=row.get('TransactionDT', 0),
        id_01=row.get('id_01', 0),
        id_02=row.get('id_02', 0)
    )
    
    for col in ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 
                'DeviceInfo', 'DeviceType', 'id_31', 'id_30', 'P_emaildomain']:
        if col in row and pd.notna(row[col]):
            val = int(row[col]) if col in ['card1', 'card2', 'card3', 'card5'] else row[col]
            entity_node = f"{col}_{val}"
            
            freq = entity_frequencies.get(entity_node, 1)
            weight = 0.01 / freq if freq > hub_threshold else 1.0 / (1.0 + np.log(freq))
            
            if not G.has_node(entity_node):
                G.add_node(entity_node, node_type=col, frequency=freq)
            G.add_edge(tx_id, entity_node, relationship=col, weight=weight)

pagerank_val = nx.pagerank(G, alpha=0.85, weight='weight')

for node, data in G.nodes(data=True):
    if data.get('node_type') == 'ValTransaction':
        neighbors = list(G.neighbors(node))
        
        neighbor_communities = [
            partition.get(n, -1) for n in neighbors if n in partition
        ]
        assigned_community = max(set(neighbor_communities), key=neighbor_communities.count) if neighbor_communities else -1
        comm_fraud_rate = comm_fraud_rates.get(assigned_community, df_features['isFraud'].mean())
        
        weighted_deg = sum(G[node][n].get('weight', 1.0) for n in neighbors)
        
        val_graph_features.append({
            'TransactionID': node,
            'isFraud': data.get('is_fraud', 0),
            'TransactionAmt': data.get('amount', 0),
            'id_01': data.get('id_01', 0),
            'id_02': data.get('id_02', 0),
            'graph_degree': G.degree(node),
            'weighted_degree': weighted_deg,
            'pagerank': pagerank_val.get(node, 0),
            'community_fraud_rate': comm_fraud_rate
        })

df_val_features = pd.DataFrame(val_graph_features)

# Score Validation Set
X_val = df_val_features[feature_cols].fillna(0)
X_val_scaled = scaler.transform(X_val)

df_val_features['anomaly_score'] = iso_forest.decision_function(X_val_scaled)
df_val_features['predicted_risk'] = -df_val_features['anomaly_score']

# Evaluate Performance
auc_score = roc_auc_score(df_val_features['isFraud'], df_val_features['predicted_risk'])
print(f"\nUnseen Validation ROC-AUC (Hub-Penalized Community KG): {auc_score:.4f}")

## Unsupervised Tabular Approach 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# 1. Load data
train_transac = pd.read_csv("Dataset/train_transaction.csv")
test_transac = pd.read_csv("Dataset/test_transaction.csv")
train_id = pd.read_csv("Dataset/train_identity.csv")
test_id = pd.read_csv("Dataset/test_identity.csv")

# 2. Merge and Split Data Chronologically
full_train_df = pd.merge(train_transac, train_id, on='TransactionID', how='left')
full_train_df = full_train_df.sort_values('TransactionDT').reset_index(drop=True)

split_index = int(len(full_train_df) * 0.8)
df_train = full_train_df.iloc[:split_index].copy()
df_val = full_train_df.iloc[split_index:].copy()

# 3. Define Tabular-Only Features (including numeric features only, dropping strings like 'discover')
feature_cols = [
    'TransactionAmt', 
    'id_01', 
    'id_02',
    'card1',
    'card2', 
    'card3', 
    'card4', 
    'card5', 
    'card6', 
    'DeviceInfo', 
    'DeviceType', 
    'id_31', 
    'id_30', 
    'P_emaildomain'
]

# 4. Prepare Training Matrix and ensure all features are purely numeric
X_train = df_train[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
y_train = df_train['isFraud']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 5. Train Unsupervised Isolation Forest
iso_forest = IsolationForest(
    n_estimators=150,
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train_scaled)

# 6. Score Validation Set
X_val = df_val[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
X_val_scaled = scaler.transform(X_val)

df_val_features['anomaly_score'] = iso_forest.decision_function(X_val_scaled)
df_val_features['unsupervised_tabular_risk'] = -df_val_features['anomaly_score']

# 7. Evaluate Performance
auc_score = roc_auc_score(df_val['isFraud'], df_val_features['unsupervised_tabular_risk'])
print(f"\nUnseen Validation ROC-AUC (Tabular-Only Baseline): {auc_score:.4f}")

## Supervised Approach 

In [24]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# 1. Load data
train_transac = pd.read_csv("Dataset/train_transaction.csv")
test_transac = pd.read_csv("Dataset/test_transaction.csv")
train_id = pd.read_csv("Dataset/train_identity.csv")
test_id = pd.read_csv("Dataset/train_identity.csv")

# 2. Merge and Split Data Chronologically
full_train_df = pd.merge(train_transac, train_id, on='TransactionID', how='left')
full_train_df = full_train_df.sort_values('TransactionDT').reset_index(drop=True)

split_index = int(len(full_train_df) * 0.8)
df_train = full_train_df.iloc[:split_index].copy()
df_val = full_train_df.iloc[split_index:].copy()

# 3. Define Features
feature_cols = [
    'TransactionAmt', 
    'id_01', 
    'id_02',
    'card1',
    'card2', 
    'card3', 
    'card4', 
    'card5', 
    'card6', 
    'DeviceInfo', 
    'DeviceType', 
    'id_31', 
    'id_30', 
    'P_emaildomain',
]

# 4. Properly encode categorical/string features for LightGBM
for col in feature_cols:
    if df_train[col].dtype == 'object' or pd.api.types.is_string_dtype(df_train[col]):
        df_train[col] = df_train[col].fillna('Missing').astype('category')
        df_val[col] = df_val[col].fillna('Missing').astype('category')
    else:
        # Fill numeric NaNs
        df_train[col] = df_train[col].fillna(0)
        df_val[col] = df_val[col].fillna(0)

X_train = df_train[feature_cols]
y_train = df_train['isFraud']

X_val = df_val[feature_cols]
y_val = df_val['isFraud']

# 5. Train Supervised LightGBM Classifier
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'seed': 42,
    'verbose': -1
}

print("Training Supervised LightGBM model...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# 6. Score Validation Set
df_val['supervised_tabular_risk'] = model.predict(X_val, num_iteration=model.best_iteration)

# 7. Evaluate Performance
auc_score = roc_auc_score(y_val, df_val['supervised_tabular_risk'])
print(f"\nUnseen Validation ROC-AUC (Supervised LightGBM Baseline): {auc_score:.4f}")

Training Supervised LightGBM model...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[207]	valid_0's auc: 0.789071

Unseen Validation ROC-AUC (Supervised LightGBM Baseline): 0.7891


## Supervised + KG

In [25]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# 1. Merge raw tabular attributes back into your graph feature dataframes
# This ensures LightGBM has access to card, device, and email columns alongside the KG metrics
raw_cols_to_merge = [
    'TransactionID', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 
    'DeviceInfo', 'DeviceType', 'id_31', 'id_30', 'P_emaildomain'
]

df_train_full = pd.merge(df_features, df_train[raw_cols_to_merge], on='TransactionID', how='left')
df_val_full = pd.merge(df_val_features, df_val[raw_cols_to_merge], on='TransactionID', how='left')

# 2. Define Combined Feature Set (Raw Tabular + KG Topological Features)
feature_cols = [
    'TransactionAmt', 
    'id_01', 
    'id_02',
    'card1',
    'card2', 
    'card3', 
    'card4', 
    'card5', 
    'card6', 
    'DeviceInfo', 
    'DeviceType', 
    'id_31', 
    'id_30', 
    'P_emaildomain',
    # Knowledge Graph topological features extracted from your network pipeline:
    'graph_degree',
    'weighted_degree',
    'pagerank',
    'community_fraud_rate'
]

# 3. Properly encode string/object columns as pandas categorical types for LightGBM
for col in feature_cols:
    if col in df_train_full.columns:
        if df_train_full[col].dtype == 'object' or pd.api.types.is_string_dtype(df_train_full[col]):
            df_train_full[col] = df_train_full[col].fillna('Missing').astype('category')
            df_val_full[col] = df_val_full[col].fillna('Missing').astype('category')
        else:
            df_train_full[col] = df_train_full[col].fillna(0)
            df_val_full[col] = df_val_full[col].fillna(0)

# 4. Prepare Training and Validation Matrices
X_train = df_train_full[[c for c in feature_cols if c in df_train_full.columns]]
y_train = df_train_full['isFraud']

X_val = df_val_full[[c for c in feature_cols if c in df_val_full.columns]]
y_val = df_val_full['isFraud']

# 5. Configure and Train Supervised LightGBM Classifier
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'seed': 42,
    'verbose': -1
}

print("Training Supervised LightGBM model with Tabular + Knowledge Graph features...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# 6. Score Validation Set and Evaluate
df_val_full['supervised_kg_tabular_risk'] = model.predict(X_val, num_iteration=model.best_iteration)
auc_score = roc_auc_score(y_val, df_val_full['supervised_kg_tabular_risk'])

print(f"\nUnseen Validation ROC-AUC (Supervised Tabular + KG Features): {auc_score:.4f}")

Training Supervised LightGBM model with Tabular + Knowledge Graph features...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[209]	valid_0's auc: 0.791577

Unseen Validation ROC-AUC (Supervised Tabular + KG Features): 0.7916
